In [ ]:
"""
Problem:
    min_{x in [0,1], y in {0,1}}  (x - 0.35)^2 + 0.2 y
    s.t.                          x + 0.8 y <= 1.0

Reformulation:
    introduce slack s in [0,1] such that
        x + 0.8 y + s = 1.0,  s >= 0
    and penalize equality with lambda * (x + 0.8 y + s - 1)^2

Encoding:
    x and s use SBE decimal encoding (weights 1,2,3,3 plus tail bit)
    y is a single binary variable

Rolling precision:
    refine J for x (and optionally s) from J0 to Jmax by Jstep

"""

import time
from dataclasses import dataclass
from typing import Dict, Tuple, List, Any

import dimod
from neal import SimulatedAnnealingSampler

DEC_WEIGHTS = (1, 2, 3, 3)  # SBE digit weights


# SBE encoding

def sbe_affine_bits(var: str, J: int, L=0.0, U=1.0) -> Tuple[float, Dict[str, float]]:
    """
    Returns affine form x = const + sum_i coeff_i * z_i for SBE-decimal encoding.
    """
    if J < 1:
        raise ValueError("J must be >= 1")

    const = L
    scale = (U - L)

    coeffs: Dict[str, float] = {}
    for j in range(1, J + 1):
        place = 10 ** (-j)
        for k, w in enumerate(DEC_WEIGHTS, start=1):
            b = f"z_{var}_{j}_{k}"
            coeffs[b] = coeffs.get(b, 0.0) + scale * place * w

    tail = f"z_{var}_tail_J{J}"
    coeffs[tail] = coeffs.get(tail, 0.0) + scale * (10 ** (-J))
    return const, coeffs


def decode_from_affine(sample: Dict[str, int], const: float, coeffs: Dict[str, float]) -> float:
    val = const
    for b, a in coeffs.items():
        val += a * float(sample.get(b, 0))
    return val


# QUBO builder

def add_linear(Qlin: Dict[str, float], v: str, w: float):
    Qlin[v] = Qlin.get(v, 0.0) + w


def add_quad(Qquad: Dict[Tuple[str, str], float], u: str, v: str, w: float):
    if u == v:
        raise ValueError("Use add_linear for diagonal terms; quadratic dict should be off-diagonal only.")
    a, b = (u, v) if u < v else (v, u)
    Qquad[(a, b)] = Qquad.get((a, b), 0.0) + w


def add_square_of_affine(
    Qlin: Dict[str, float],
    Qquad: Dict[Tuple[str, str], float],
    offset_ref: List[float],
    c0: float,
    coeffs: Dict[str, float],
    weight: float = 1.0,
):
    """
    Add weight * (c0 + sum_i a_i z_i)^2 to QUBO (BQM).
    Uses z_i^2 = z_i, so linear gets a_i^2 + 2*c0*a_i; quadratic gets 2*a_i*a_j.
    """
    offset_ref[0] += weight * (c0 * c0)

    items = list(coeffs.items())

    # linear: weight*(a_i^2 + 2*c0*a_i)*z_i
    for bi, ai in items:
        add_linear(Qlin, bi, weight * (ai * ai + 2.0 * c0 * ai))

    # quadratic: weight*(2*a_i*a_j)*z_i*z_j
    for i in range(len(items)):
        bi, ai = items[i]
        for j in range(i + 1, len(items)):
            bj, aj = items[j]
            add_quad(Qquad, bi, bj, weight * (2.0 * ai * aj))


def qubo_stats(bqm: dimod.BinaryQuadraticModel) -> Dict[str, int]:
    return {
        "n_vars": len(bqm.variables),
        "n_linear": len(bqm.linear),
        "n_quadratic": len(bqm.quadratic),
    }


# Solver + rolling precision

@dataclass
class SAOptions:
    num_reads: int = 200
    sweeps: int = 2500
    seed: int = 13


def solve_example2_miqp_at_precision(
    Jx: int,
    Js: int,
    penalty_lambda: float,
    sa: SAOptions,
) -> Dict[str, Any]:
    """
    Build and solve QUBO for a fixed precision (Jx for x, Js for slack s).
    """
    sampler = SimulatedAnnealingSampler()

    # Decision variables
    # x in [0,1] via SBE
    cx, ax = sbe_affine_bits("x", Jx, 0.0, 1.0)
    # slack s in [0,1] via SBE
    cs, as_ = sbe_affine_bits("s", Js, 0.0, 1.0)
    # y is a single binary variable
    yname = "y"

    Qlin: Dict[str, float] = {}
    Qquad: Dict[Tuple[str, str], float] = {}
    offset = [0.0]

    # Objective term: (x - 0.35)^2
    add_square_of_affine(Qlin, Qquad, offset, cx - 0.35, ax, weight=1.0)

    # Objective term: 0.2*y
    add_linear(Qlin, yname, 0.2)

    # Constraint: x + 0.8*y + s = 1.0  (slack enforces <=)
    # Build affine g = (cx + 0.8*0 + cs - 1) + sum(ax*zx) + 0.8*y + sum(as*zs)
    g0 = (cx + cs - 1.0)
    gcoeffs: Dict[str, float] = {}

    # x bits
    for b, a in ax.items():
        gcoeffs[b] = gcoeffs.get(b, 0.0) + a
    # s bits
    for b, a in as_.items():
        gcoeffs[b] = gcoeffs.get(b, 0.0) + a
    # y bit
    gcoeffs[yname] = gcoeffs.get(yname, 0.0) + 0.8

    add_square_of_affine(Qlin, Qquad, offset, g0, gcoeffs, weight=penalty_lambda)

    bqm = dimod.BinaryQuadraticModel(Qlin, Qquad, offset[0], vartype=dimod.BINARY)

    t0 = time.perf_counter()
    ss = sampler.sample(bqm, num_reads=sa.num_reads, sweeps=sa.sweeps, seed=sa.seed)
    t1 = time.perf_counter()

    best = ss.first.sample
    energy = float(ss.first.energy)

    x = decode_from_affine(best, cx, ax)
    s = decode_from_affine(best, cs, as_)
    y = int(best.get(yname, 0))

    # True objective (no penalty)
    obj = (x - 0.35) ** 2 + 0.2 * y

    # Inequality check (original): x + 0.8 y <= 1
    viol = max(0.0, x + 0.8 * y - 1.0)

    # Equality residual (penalty target): x + 0.8 y + s - 1 = 0
    resid = (x + 0.8 * y + s - 1.0)

    return {
        "Jx": Jx,
        "Js": Js,
        "x": x,
        "y": y,
        "s": s,
        "obj": obj,
        "viol": viol,
        "resid": resid,
        "pen_energy": energy,
        "time_s": (t1 - t0),
        "stats": qubo_stats(bqm),
    }


def rolling_precision_example2_miqp_backtracking(
    J0_x: int = 1,
    J0_s: int = 1,
    Jstep: int = 1,
    Jmax_x: int = 4,
    Jmax_s: int = 4,
    penalty_lambda: float = 100.0,
    sa: SAOptions = SAOptions(num_reads=200, sweeps=2500, seed=13),
    theta: float = 1e-10,          # acceptance threshold on penalized energy
    max_iters: int = 50,
) -> Dict[str, Any]:
    """
    Rolling precision with backtracking for Example 2.

    State is the precision vector (Jx, Js). Moves:
      - refine x: (Jx+Jstep, Js)
      - refine s: (Jx, Js+Jstep)
      - backtrack x: (Jx-Jstep, Js)
      - backtrack s: (Jx, Js-Jstep)

    Accept any move only if penalized energy improves by at least theta.
    Never revisit a (Jx, Js) already evaluated.
    """

    # --- clamp feasibility of candidate J ---
    def valid(Jx: int, Js: int) -> bool:
        if Jx < J0_x or Js < J0_s:
            return False
        if Jx > Jmax_x or Js > Jmax_s:
            return False
        return True

    def J_key(Jx: int, Js: int) -> Tuple[int, int]:
        return (Jx, Js)

    # --- initial solve ---
    Jx, Js = J0_x, J0_s
    visited = {J_key(Jx, Js)}

    best = solve_example2_miqp_at_precision(Jx, Js, penalty_lambda, sa)
    best_pen = best["pen_energy"]   # F* (penalized objective on the grid)

    history: List[Dict[str, Any]] = []
    history.append({"iter": 0, "move": "init", "var": None, **best})

    # stack of accepted states 
    stack: List[Tuple[int, int, float, Dict[str, Any]]] = [(Jx, Js, best_pen, best)]

    improved = True
    it = 0

    while improved and it < max_iters:
        it += 1
        improved = False

        # Candidate moves (refinements first, then backtracks)
        candidates: List[Tuple[str, str, int, int]] = []

        # --- refinements ---
        candidates.append(("refine", "x",    Jx + Jstep, Js))
        candidates.append(("refine", "s",    Jx, Js + Jstep))

        # coupled refine
        candidates.append(("refine", "xs",   Jx + Jstep, Js + Jstep))

        # --- backtracks ---
        candidates.append(("backtrack", "x", Jx - Jstep, Js))
        candidates.append(("backtrack", "s", Jx, Js - Jstep))

        # coupled backtrack
        candidates.append(("backtrack", "xs", Jx - Jstep, Js - Jstep))


        # evaluate unvisited candidates; accept first improving candidate
        for move_type, var, Jx_c, Js_c in candidates:
            if not valid(Jx_c, Js_c):
                continue
            key = J_key(Jx_c, Js_c)
            if key in visited:
                continue
            visited.add(key)

            cand = solve_example2_miqp_at_precision(Jx_c, Js_c, penalty_lambda, sa)
            cand_pen = cand["pen_energy"]

            # acceptance criterion: improve by at least theta (monotone incumbent)
            if cand_pen <= best_pen - theta:
                # accept
                Jx, Js = Jx_c, Js_c
                best = cand
                best_pen = cand_pen
                improved = True

                history.append({"iter": it, "move": move_type, "var": var, **cand})
                stack.append((Jx, Js, best_pen, best))
                break  # go to next outer iteration

        # if no candidate improved, loop terminates

    # for comparison: monolithic solve at max precisions
    monolithic = solve_example2_miqp_at_precision(Jmax_x, Jmax_s, penalty_lambda, sa)

    return {
        "history": history,
        "best": best,
        "best_penalized": best_pen,
        "best_J": {"Jx": Jx, "Js": Js},
        "visited_count": len(visited),
        "monolithic": monolithic,
    }


if __name__ == "__main__":
    report = rolling_precision_example2_miqp_backtracking(
        J0_x=1,
        J0_s=1,
        Jstep=1,
        Jmax_x=4,
        Jmax_s=4,
        penalty_lambda=100.0,
        sa=SAOptions(num_reads=200, sweeps=2500, seed=13),
        theta=1e-10,
        max_iters=50,
    )

    print("\n--- Rolling precision WITH backtracking (Example 2) ---")
    for rec in report["history"]:
        st = rec["stats"]
        print(
            f"it={rec['iter']:02d} move={rec['move']:<9} var={str(rec.get('var')):<2} "
            f"Jx={rec['Jx']} Js={rec['Js']} "
            f"x={rec['x']:.6f} y={rec['y']} s={rec['s']:.6f} "
            f"obj={rec['obj']:.3e} penE={rec['pen_energy']:.6e} "
            f"viol={rec['viol']:.1e} resid={rec['resid']:+.1e} "
            f"nvars={st['n_vars']} nquad={st['n_quadratic']} time={rec['time_s']:.3f}s"
        )

    m = report["monolithic"]
    st = m["stats"]
    print("\n--- Monolithic (max J) ---")
    print(
        f"Jx={m['Jx']} Js={m['Js']} "
        f"x={m['x']:.6f} y={m['y']} s={m['s']:.6f} "
        f"obj={m['obj']:.3e} penE={m['pen_energy']:.6e} "
        f"viol={m['viol']:.1e} resid={m['resid']:+.1e} "
        f"nvars={st['n_vars']} nquad={st['n_quadratic']} time={m['time_s']:.3f}s"
    )




--- Rolling precision WITH backtracking (Example 2) ---
it=00 move=init      var=None Jx=1 Js=1 x=0.400000 y=0 s=0.600000 obj=2.500e-03 penE=2.500000e-03 viol=0.0e+00 resid=+0.0e+00 nvars=11 nquad=55 time=0.028s
it=01 move=refine    var=xs Jx=2 Js=2 x=0.350000 y=0 s=0.650000 obj=1.233e-32 penE=-7.815970e-14 viol=0.0e+00 resid=+2.2e-16 nvars=19 nquad=171 time=0.040s

--- Monolithic (max J) ---
Jx=4 Js=4 x=0.350000 y=0 s=0.650000 obj=1.233e-32 penE=-1.563194e-13 viol=0.0e+00 resid=+2.2e-16 nvars=35 nquad=595 time=0.082s
